# 01 — Generate Synthetic Brake Testbench Data

This notebook generates **realistic synthetic brake testbench sensor data** for the Cycle Detection demo.

The data simulates continuous time-series readings from brake testbenches for the **Pressure Control Unit (PCU)**. Each test run contains alternating idle and brake-cycle segments, with ground truth labels (`cycle_status`) indicating whether each data point belongs to a brake cycle.

### Data Schema
| Column | Type | Description |
|--------|------|-------------|
| `timestamp` | timestamp | Measurement time (100ms resolution) |
| `testbench_id` | string | Testbench identifier |
| `test_run_id` | string | Unique test run identifier |
| `pressure_bar` | double | Brake pressure in bar |
| `valve_state` | int | Magnet valve state (0=closed, 1=open) |
| `temperature_c` | double | Component temperature in °C |
| `flow_rate` | double | Hydraulic flow rate (l/min) |
| `cycle_status` | int | Ground truth label — 1 if inside a brake cycle, 0 if idle |

In [ ]:
%run ./00_Config

In [ ]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Infrastructure ready: {CATALOG}.{SCHEMA}")

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

SEED = 42
np.random.seed(SEED)

# --- Testbench and run configuration ---
TESTBENCHES = ["TB-PCU-001", "TB-PCU-002", "TB-PCU-003", "TB-PCU-004", "TB-PCU-005"]
RUNS_PER_TESTBENCH = 8
SAMPLING_INTERVAL_MS = 100  # 10 Hz

# --- Cycle shape parameters ---
IDLE_DURATION_RANGE = (30, 80)      # idle data points between cycles (3-8 sec)
RAMP_UP_RANGE = (20, 40)            # ramp-up phase (2-4 sec)
HOLD_RANGE = (50, 120)              # hold phase (5-12 sec)
RAMP_DOWN_RANGE = (15, 30)          # ramp-down phase (1.5-3 sec)
CYCLES_PER_RUN_RANGE = (8, 18)      # number of brake cycles per test run

TARGET_PRESSURE_MEAN = 7.5          # bar
TARGET_PRESSURE_STD = 1.2           # bar
AMBIENT_TEMP = 22.0                 # °C
TEMP_RISE_PER_CYCLE = 0.4           # °C per cycle

In [ ]:
def generate_single_run(testbench_id: str, run_idx: int, base_time: datetime) -> pd.DataFrame:
    """Generate one test run with alternating idle and brake-cycle segments."""
    run_id = f"{testbench_id}-RUN-{run_idx:03d}"
    n_cycles = np.random.randint(*CYCLES_PER_RUN_RANGE)

    rows = []
    t_idx = 0
    temp = AMBIENT_TEMP + np.random.normal(0, 0.5)

    for cycle_num in range(n_cycles):
        # --- Idle segment (not a cycle) ---
        idle_len = np.random.randint(*IDLE_DURATION_RANGE)
        for _ in range(idle_len):
            ts = base_time + timedelta(milliseconds=t_idx * SAMPLING_INTERVAL_MS)
            noise = np.random.normal(0, 0.05)
            rows.append({
                "timestamp": ts,
                "testbench_id": testbench_id,
                "test_run_id": run_id,
                "pressure_bar": round(max(0, 0.1 + noise), 3),
                "valve_state": 0,
                "temperature_c": round(temp + np.random.normal(0, 0.1), 2),
                "flow_rate": round(max(0, np.random.normal(0.2, 0.05)), 3),
                "cycle_status": 0,
            })
            t_idx += 1

        # --- Brake cycle: ramp-up → hold → ramp-down ---
        ramp_up_len = np.random.randint(*RAMP_UP_RANGE)
        hold_len = np.random.randint(*HOLD_RANGE)
        ramp_down_len = np.random.randint(*RAMP_DOWN_RANGE)
        target_p = max(3.0, np.random.normal(TARGET_PRESSURE_MEAN, TARGET_PRESSURE_STD))

        # Ramp up
        for i in range(ramp_up_len):
            ts = base_time + timedelta(milliseconds=t_idx * SAMPLING_INTERVAL_MS)
            frac = i / ramp_up_len
            p = frac * target_p + np.random.normal(0, 0.08)
            temp += 0.005
            rows.append({
                "timestamp": ts,
                "testbench_id": testbench_id,
                "test_run_id": run_id,
                "pressure_bar": round(max(0, p), 3),
                "valve_state": 1,
                "temperature_c": round(temp + np.random.normal(0, 0.1), 2),
                "flow_rate": round(max(0, np.random.normal(2.5, 0.3) * frac), 3),
                "cycle_status": 1,
            })
            t_idx += 1

        # Hold
        for i in range(hold_len):
            ts = base_time + timedelta(milliseconds=t_idx * SAMPLING_INTERVAL_MS)
            p = target_p + np.random.normal(0, 0.12)
            temp += 0.002
            rows.append({
                "timestamp": ts,
                "testbench_id": testbench_id,
                "test_run_id": run_id,
                "pressure_bar": round(max(0, p), 3),
                "valve_state": 1,
                "temperature_c": round(temp + np.random.normal(0, 0.1), 2),
                "flow_rate": round(max(0, np.random.normal(0.5, 0.1)), 3),
                "cycle_status": 1,
            })
            t_idx += 1

        # Ramp down
        for i in range(ramp_down_len):
            ts = base_time + timedelta(milliseconds=t_idx * SAMPLING_INTERVAL_MS)
            frac = 1.0 - (i / ramp_down_len)
            p = frac * target_p + np.random.normal(0, 0.08)
            temp -= 0.003
            rows.append({
                "timestamp": ts,
                "testbench_id": testbench_id,
                "test_run_id": run_id,
                "pressure_bar": round(max(0, p), 3),
                "valve_state": 1,
                "temperature_c": round(temp + np.random.normal(0, 0.1), 2),
                "flow_rate": round(max(0, np.random.normal(2.5, 0.3) * frac), 3),
                "cycle_status": 1,
            })
            t_idx += 1

        temp += TEMP_RISE_PER_CYCLE * np.random.uniform(0.5, 1.5)

    # Trailing idle
    for _ in range(np.random.randint(20, 50)):
        ts = base_time + timedelta(milliseconds=t_idx * SAMPLING_INTERVAL_MS)
        rows.append({
            "timestamp": ts,
            "testbench_id": testbench_id,
            "test_run_id": run_id,
            "pressure_bar": round(max(0, 0.1 + np.random.normal(0, 0.05)), 3),
            "valve_state": 0,
            "temperature_c": round(temp + np.random.normal(0, 0.1), 2),
            "flow_rate": round(max(0, np.random.normal(0.2, 0.05)), 3),
            "cycle_status": 0,
        })
        t_idx += 1

    return pd.DataFrame(rows)

In [ ]:
all_runs = []
base = datetime(2025, 9, 1, 6, 0, 0)

for tb in TESTBENCHES:
    for run_idx in range(RUNS_PER_TESTBENCH):
        run_base = base + timedelta(days=run_idx * np.random.randint(2, 5),
                                    hours=np.random.randint(0, 8))
        df_run = generate_single_run(tb, run_idx, run_base)
        all_runs.append(df_run)
        print(f"  {tb} / RUN-{run_idx:03d}: {len(df_run):,} rows, "
              f"{df_run['cycle_status'].sum():,} cycle points "
              f"({df_run['cycle_status'].mean()*100:.1f}%)")

pdf = pd.concat(all_runs, ignore_index=True)
print(f"\nTotal: {len(pdf):,} rows across {len(TESTBENCHES)} testbenches, "
      f"{RUNS_PER_TESTBENCH} runs each")
print(f"Cycle ratio: {pdf['cycle_status'].mean()*100:.1f}%")

In [ ]:
sdf = spark.createDataFrame(pdf)
sdf.write.mode("overwrite").saveAsTable(RAW_SENSOR_TABLE)

row_count = spark.table(RAW_SENSOR_TABLE).count()
print(f"Saved {row_count:,} rows to {RAW_SENSOR_TABLE}")

In [ ]:
display(spark.table(RAW_SENSOR_TABLE).limit(20))

### Quick Visualization — Single Test Run
Plot pressure and valve state for one run to visually verify the cycle structure.

In [ ]:
import matplotlib.pyplot as plt

sample_run = pdf[pdf["test_run_id"] == pdf["test_run_id"].unique()[0]].copy()
sample_run["seconds"] = (sample_run["timestamp"] - sample_run["timestamp"].min()).dt.total_seconds()

fig, axes = plt.subplots(3, 1, figsize=(16, 8), sharex=True)

# Shade cycle regions
for ax in axes:
    cycle_mask = sample_run["cycle_status"]
    start = None
    for i, (sec, is_c) in enumerate(zip(sample_run["seconds"], cycle_mask)):
        if is_c and start is None:
            start = sec
        elif not is_c and start is not None:
            ax.axvspan(start, sec, alpha=0.15, color="green")
            start = None
    if start is not None:
        ax.axvspan(start, sample_run["seconds"].iloc[-1], alpha=0.15, color="green")

axes[0].plot(sample_run["seconds"], sample_run["pressure_bar"], linewidth=0.5, color="#1f77b4")
axes[0].set_ylabel("Pressure (bar)")
axes[0].set_title(f"Test Run: {sample_run['test_run_id'].iloc[0]}  |  Green = Brake Cycle")

axes[1].plot(sample_run["seconds"], sample_run["valve_state"], linewidth=0.8, color="#d62728", drawstyle="steps-post")
axes[1].set_ylabel("Valve State")
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(["Closed", "Open"])

axes[2].plot(sample_run["seconds"], sample_run["temperature_c"], linewidth=0.5, color="#ff7f0e")
axes[2].set_ylabel("Temperature (°C)")
axes[2].set_xlabel("Time (seconds)")

plt.tight_layout()
plt.show()